# Inside `nn.Module`

You have subclassed `nn.Module` many times. This notebook opens it up, so that surprising behaviour (a layer that never trains, a tensor that doesn't move to the GPU, a checkpoint that won't load) becomes predictable.

1. **Parameters vs buffers vs plain tensors**: what gets registered
2. **Containers**: `Sequential`, `ModuleList`, `ModuleDict`, and the plain-list trap
3. **Walking a model**: `parameters`, `named_modules`, `children`, `apply`
4. **`state_dict`**: what is saved, partial loading, key mismatches
5. **Freezing** parameters, and optimizers
6. **Hooks**: look inside a forward/backward pass without editing the model
7. **`train()` / `eval()`** propagation
8. A **custom layer** and a custom **autograd Function**

All CPU, runs in seconds.

In [1]:
import copy
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0)

## 1. Parameters, buffers, plain tensors

`nn.Module.__setattr__` watches what you assign in `__init__`:

| Assigned | Registered as | In `parameters()` | In `state_dict` | Moved by `.to()` | Trained |
|---|---|---|---|---|---|
| `nn.Parameter(...)` | parameter | yes | yes | yes | yes |
| `register_buffer(name, t)` | buffer | no | yes (unless `persistent=False`) | yes | no |
| `self.t = torch.tensor(...)` | nothing | no | no | **no** | no |
| `nn.Module` instance | submodule | via child | via child | yes | via child |

In [2]:
class Demo(nn.Module):
    def __init__(self):
        super().__init__()
        self.w      = nn.Parameter(torch.randn(3))              # learned
        self.register_buffer('running', torch.zeros(3))         # state, not learned
        self.plain  = torch.ones(3)                             # invisible to the module
        self.lin    = nn.Linear(3, 1)                           # submodule
    def forward(self, x):
        return self.lin(x * self.w + self.running + self.plain)

m = Demo()
print('parameters :', [n for n, _ in m.named_parameters()])
print('buffers    :', [n for n, _ in m.named_buffers()])
print('state_dict :', list(m.state_dict().keys()))
print("'plain' in state_dict:", 'plain' in m.state_dict())

parameters : ['w', 'lin.weight', 'lin.bias']
buffers    : ['running']
state_dict : ['w', 'running', 'lin.weight', 'lin.bias']
'plain' in state_dict: False


**Why buffers exist.** BatchNorm's running mean/variance must be saved with the model and moved to the GPU with it, but must not receive gradients. That is a buffer. The positional-encoding table in the Transformer notebook is another (`persistent=False` would also keep it out of checkpoints).

The silent bug is the third row: a plain tensor attribute works on CPU, then crashes (or worse, silently stays on the wrong device) after `.to('cuda')`.

In [3]:
# Demonstrate with dtype, since it behaves like device: .to() converts registered tensors only
m64 = Demo().to(torch.float64)
print('param  dtype:', m64.w.dtype)
print('buffer dtype:', m64.running.dtype)
print('plain  dtype:', m64.plain.dtype, '  <- left behind')

param  dtype: torch.float64
buffer dtype: torch.float64
plain  dtype: torch.float32   <- left behind


## 2. Containers, and the plain-list trap

Submodules stored in a plain Python `list` or `dict` are **not registered**. The model runs, but those layers are missing from `parameters()`, so the optimizer never updates them, and they are missing from `state_dict()`, so they are never saved.

In [4]:
class Bad(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(4, 4) for _ in range(3)]       # plain list
    def forward(self, x):
        for l in self.layers: x = l(x)
        return x

class Good(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList(nn.Linear(4, 4) for _ in range(3))
    def forward(self, x):
        for l in self.layers: x = l(x)
        return x

count = lambda m: sum(p.numel() for p in m.parameters())
print('plain list : params =', count(Bad()),  ' state_dict keys =', len(Bad().state_dict()))
print('ModuleList : params =', count(Good()), ' state_dict keys =', len(Good().state_dict()))

plain list : params = 0  state_dict keys = 0
ModuleList : params = 60  state_dict keys = 6


- **`nn.Sequential`**: layers run in order; use when `forward` is just a chain.
- **`nn.ModuleList`**: a registered list; you write the `forward` loop (skip connections, early exit).
- **`nn.ModuleDict`**: a registered dict; pick a branch by name (multi-task heads, switchable blocks). Same trap applies to dicts.

In [5]:
heads = nn.ModuleDict({'cls': nn.Linear(8, 3), 'reg': nn.Linear(8, 1)})
x = torch.randn(2, 8)
print({k: tuple(h(x).shape) for k, h in heads.items()})
print('registered keys:', [k for k in heads.state_dict()][:2], '...')

{'cls': (2, 3), 'reg': (2, 1)}
registered keys: ['cls.weight', 'cls.bias'] ...


## 3. Walking a model

In [6]:
net = nn.Sequential(
    nn.Conv2d(1, 8, 3), nn.BatchNorm2d(8), nn.ReLU(),
    nn.Flatten(), nn.Linear(8 * 26 * 26, 10))

print('children() (direct):')
for name, mod in net.named_children(): print(f'  {name}: {mod.__class__.__name__}')

print('\nparameters by layer:')
for name, p in net.named_parameters(): print(f'  {name:12s} {tuple(p.shape)!s:16s} requires_grad={p.requires_grad}')

print('\ntotal params:', sum(p.numel() for p in net.parameters()))

children() (direct):
  0: Conv2d
  1: BatchNorm2d
  2: ReLU
  3: Flatten
  4: Linear

parameters by layer:
  0.weight     (8, 1, 3, 3)     requires_grad=True
  0.bias       (8,)             requires_grad=True
  1.weight     (8,)             requires_grad=True
  1.bias       (8,)             requires_grad=True
  4.weight     (10, 5408)       requires_grad=True
  4.bias       (10,)            requires_grad=True

total params: 54186


`children()` gives direct children only; `modules()` / `named_modules()` recurse into nested modules (and include the model itself). `apply(fn)` calls `fn` on every submodule, the standard way to initialize weights:

In [7]:
def init_fn(m):
    if isinstance(m, (nn.Linear, nn.Conv2d)):
        nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
        nn.init.zeros_(m.bias)

net.apply(init_fn)
print('bias all zero after apply:', bool((net[0].bias == 0).all()))

bias all zero after apply: True


## 4. `state_dict`

An ordered dict of name -> tensor for parameters **and** persistent buffers. The names come from the attribute path (`layers.0.weight`). Loading matches by name and shape, so renaming an attribute breaks old checkpoints.

In [8]:
print({k: tuple(v.shape) for k, v in net.state_dict().items()})

{'0.weight': (8, 1, 3, 3), '0.bias': (8,), '1.weight': (8,), '1.bias': (8,), '1.running_mean': (8,), '1.running_var': (8,), '1.num_batches_tracked': (), '4.weight': (10, 5408), '4.bias': (10,)}


In [9]:
# Note the BatchNorm buffers: running_mean, running_var, num_batches_tracked
sd = net.state_dict()
print('BN buffers saved:', [k for k in sd if 'running' in k or 'num_batches' in k])

# Partial loading: strict=False reports mismatches instead of raising
src = nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Linear(4, 2))
dst = nn.Sequential(nn.Linear(4, 4), nn.ReLU(), nn.Linear(4, 5))       # different head
sd = {k: v for k, v in src.state_dict().items() if not k.startswith('2.')}   # drop the old head
result = dst.load_state_dict(sd, strict=False)
print('missing   :', result.missing_keys)
print('unexpected:', result.unexpected_keys)
print('body copied:', torch.equal(src[0].weight, dst[0].weight))

BN buffers saved: ['1.running_mean', '1.running_var', '1.num_batches_tracked']
missing   : ['2.weight', '2.bias']
unexpected: []
body copied: True


This is the mechanism behind transfer learning: load a pretrained body, leave the new head at its fresh init. `strict=False` is also dangerous: **always read `missing_keys`**, since a typo'd prefix loads nothing and raises nothing.

Keys pick up a `module.` prefix when a model was saved wrapped in `nn.DataParallel`; strip it before loading into an unwrapped model.

## 5. Freezing, and what the optimizer sees

`requires_grad=False` stops gradient computation for a parameter. Two common patterns:

In [10]:
model = nn.Sequential(nn.Linear(10, 32), nn.ReLU(), nn.Linear(32, 2))

for p in model[0].parameters(): p.requires_grad = False                # freeze the body

trainable = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(trainable, lr=0.1)                              # hand the optimizer only trainable params
print('trainable params:', sum(p.numel() for p in trainable), 'of', sum(p.numel() for p in model.parameters()))

before = model[0].weight.clone(), model[2].weight.clone()
loss = model(torch.randn(8, 10)).pow(2).mean(); loss.backward(); opt.step()
print('frozen layer changed :', not torch.equal(before[0], model[0].weight))
print('trainable layer changed:', not torch.equal(before[1], model[2].weight))
print('frozen grad is None  :', model[0].weight.grad is None)

trainable params: 66 of 418
frozen layer changed : False
trainable layer changed: True
frozen grad is None  : True


Notes:
- Frozen parameters have `.grad is None`; no memory is spent on their gradients.
- Passing frozen parameters to the optimizer is harmless (they have no grad, so they are skipped) but filtering is clearer.
- **Freezing does not put BatchNorm/Dropout in eval mode.** A frozen BatchNorm in `train()` mode still updates its running statistics. Call `.eval()` on frozen blocks too (or on the whole backbone) when using one as a fixed feature extractor.

In [11]:
bn = nn.BatchNorm1d(4)
for p in bn.parameters(): p.requires_grad = False
before = bn.running_mean.clone()
bn.train(); bn(torch.randn(16, 4) + 5)
print('frozen BN in train mode: running_mean changed =', not torch.equal(before, bn.running_mean))
bn.eval(); before = bn.running_mean.clone(); bn(torch.randn(16, 4) + 5)
print('frozen BN in eval mode : running_mean changed =', not torch.equal(before, bn.running_mean))

frozen BN in train mode: running_mean changed = True
frozen BN in eval mode : running_mean changed = False


## 6. Hooks

Hooks let you observe or modify a layer's inputs, outputs or gradients **without editing the model's code**. Typical uses: extract intermediate features from a pretrained network, log activation statistics, debug exploding gradients (compare with the debugging notebook).

- `register_forward_hook(fn(module, input, output))`
- `register_full_backward_hook(fn(module, grad_input, grad_output))`
- `Tensor.register_hook(fn(grad))`

Each returns a handle; call `handle.remove()` when done.

In [12]:
net = nn.Sequential(nn.Linear(10, 16), nn.ReLU(), nn.Linear(16, 3))
feats, grads = {}, {}

def save_out(name):
    def hook(mod, inp, out): feats[name] = out.detach()
    return hook
def save_grad(name):
    def hook(mod, grad_in, grad_out): grads[name] = grad_out[0].detach()
    return hook

handles  = [net[1].register_forward_hook(save_out('relu'))]
handles += [net[2].register_full_backward_hook(save_grad('fc2'))]

x = torch.randn(4, 10)
net(x).sum().backward()
print('captured relu output :', tuple(feats['relu'].shape), ' fraction zero:', (feats['relu'] == 0).float().mean().item())
print('captured fc2 grad_out:', tuple(grads['fc2'].shape))
for h in handles: h.remove()

captured relu output : (4, 16)  fraction zero: 0.5625
captured fc2 grad_out: (4, 3)


A forward hook that **returns** a value replaces the layer's output, which is a quick way to ablate a layer or inject noise for an experiment.

In [13]:
h = net[1].register_forward_hook(lambda m, i, o: o * 0)                 # zero out the ReLU output
print('output with ReLU zeroed == bias of last layer:', torch.allclose(net(x), net[2].bias.expand(4, 3)))
h.remove()

output with ReLU zeroed == bias of last layer: True


## 7. `train()` / `eval()` propagate

`model.train(mode)` sets `self.training` on the module **and every submodule recursively**. It changes the behaviour of Dropout and BatchNorm only; it does not disable gradients (that is `torch.no_grad()`), and the two are independent.

In [14]:
net = nn.Sequential(nn.Linear(4, 4), nn.Dropout(0.5), nn.BatchNorm1d(4))
net.eval()
print('after eval():', [m.training for m in net.modules()])
x = torch.randn(1000, 4)
print('eval  deterministic:', torch.equal(net(x), net(x)))
net.train()
print('train deterministic:', torch.equal(net(x), net(x)), ' (dropout is random)')
with torch.no_grad():
    out = net(x)
print('no_grad output requires_grad:', out.requires_grad, '| module still in train mode:', net.training)

after eval(): [False, False, False, False]
eval  deterministic: True
train deterministic: False  (dropout is random)
no_grad output requires_grad: False | module still in train mode: True


## 8. A custom layer and a custom autograd Function

A layer is just a Module with parameters and a `forward`. Autograd differentiates the operations inside it automatically.

In [15]:
class LayerScale(nn.Module):
    """y = gamma * x, one learned scale per feature (used in modern vision transformers)."""
    def __init__(self, dim, init=0.1):
        super().__init__()
        self.gamma = nn.Parameter(torch.full((dim,), init))
    def forward(self, x):
        return x * self.gamma
    def extra_repr(self):                                # shown in print(model)
        return f'dim={self.gamma.numel()}'

block = nn.Sequential(nn.Linear(8, 8), LayerScale(8))
print(block)
block(torch.randn(2, 8)).sum().backward()
print('gamma got a gradient:', block[1].gamma.grad is not None)

Sequential(
  (0): Linear(in_features=8, out_features=8, bias=True)
  (1): LayerScale(dim=8)
)
gamma got a gradient: True


Occasionally you need an operation autograd cannot differentiate (or want a different gradient). `torch.autograd.Function` lets you write `forward` and `backward` yourself. Example: **clip the gradient but not the value**:

```
forward:  identity
backward: clamp incoming gradient to [-c, c]
```

Always verify a custom backward with `torch.autograd.gradcheck` (needs float64).

In [16]:
class ClipGrad(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, c):
        ctx.c = c
        return x.view_as(x)
    @staticmethod
    def backward(ctx, g):
        return g.clamp(-ctx.c, ctx.c), None            # one grad per forward input (c gets None)

x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = ClipGrad.apply(x, 0.5)
(y * torch.tensor([0.1, 1.0, 10.0])).sum().backward()      # raw grads would be 0.1, 1, 10
print('value unchanged :', y.detach())
print('clipped gradient:', x.grad)

# gradcheck verifies against numerical gradients; use a smooth op for the demo
class Square(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x): ctx.save_for_backward(x); return x ** 2
    @staticmethod
    def backward(ctx, g): (x,) = ctx.saved_tensors; return 2 * x * g
xd = torch.randn(5, dtype=torch.float64, requires_grad=True)
print('gradcheck(Square):', torch.autograd.gradcheck(Square.apply, (xd,)))

value unchanged : tensor([1., 2., 3.])
clipped gradient: tensor([0.1000, 0.5000, 0.5000])
gradcheck(Square): True


## Cheat sheet

| Want | Use |
|---|---|
| Learned tensor | `nn.Parameter` |
| Saved, moved, not learned | `register_buffer` |
| A list/dict of layers | `ModuleList` / `ModuleDict`, never plain `list` / `dict` |
| Init every layer | `model.apply(fn)` |
| Save weights | `torch.save(model.state_dict(), path)` |
| Load into a changed model | `load_state_dict(sd, strict=False)`, then read `missing_keys` |
| Freeze | `requires_grad=False` **and** `.eval()` for BatchNorm/Dropout |
| Peek inside | forward / backward hooks, then `handle.remove()` |
| Custom gradient | `torch.autograd.Function` + `gradcheck` |

## Exercises
1. Use a forward hook to record the mean and std of every `Linear` output in a 5-layer MLP (compare to the debugging notebook's activation tables).
2. Make `Bad` from section 2 train: swap the list for `ModuleList` and confirm the loss now goes down.
3. Save a model, rename one attribute in the class, and load: read the error. Then fix it with a key-renaming loop over the state_dict.
4. Write a `GradReverse` Function (identity forward, negated gradient backward), the building block of domain-adversarial training.